<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 12 · Mathematical Tools
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

Modern quantitative finance relies heavily on numerical methods. This chapter
shows how to approach core mathematical tasks—approximation, optimization,
integration, and symbolic manipulation—using Python tools that you can reuse
throughout the book. You work entirely in the IPython REPL, using `NumPy`,
`SciPy`, and `SymPy` for concrete, finance-flavoured examples. Later
reference-style appendices (see
<<ch_linear_algebra_and_optimization_toolkit>>,
<<ch_probability_statistics_and_stochastic_calculus_essentials>>, and
<<ch_numerical_methods_and_simulation_notes>>) provide more systematic
toolkits; here the focus is on how to think about and use these methods in
practice.


## Approximation and Regression


Many quantitative models start from an unknown relationship between variables:
prices and risk factors, volatilities and maturities, or yields and coupons.


### Example Function and Sample Data


You begin with a one-dimensional function that mimics a smooth, slightly wavy
term structure.


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
mpl.style.use("seaborn-v0_8")
mpl.rcParams.update({"font.family": "serif", "figure.dpi": 300})
rng = np.random.default_rng(seed=42)
def term_structure(x: np.ndarray) -> np.ndarray:
    # The function `term_structure()` stands in for a smooth, slightly wavy term
    # structure that you might otherwise obtain from a pricing model.
    """Synthetic curve with linear and sinusoidal components."""
    return 0.5 * np.sin(x) + 0.1 * x
# The grid might represent ten equally spaced maturities in years.
x_grid = np.linspace(0.0, 10.0, 50)
y_clean = term_structure(x_grid)
noise = 0.05 * rng.standard_normal(x_grid.shape[0])
# The observed curve adds small random noise to mimic market imperfections.
y_obs = y_clean + noise
y_obs[:5]

### Linear Least-Squares Regression with NumPy


Ordinary least squares (OLS) fits a model of the form $y \approx X \beta$ by
minimizing the sum of squared residuals.


In [ ]:
# The first column models the intercept; the second column models a linear
# dependence
# on maturity.
X_lin = np.column_stack([np.ones_like(x_grid), x_grid])
# `np.linalg.lstsq()` returns the least-squares coefficients, residual sum
# of squares,
# matrix rank, and singular values.
beta_lin, residuals, rank, svals = np.linalg.lstsq(X_lin, y_obs, rcond=None)

In [ ]:
beta_lin

In [ ]:
# The fitted curve `y_lin` is the orthogonal projection of `y_obs` onto the
# column
# space of `X_lin`.
y_lin = X_lin @ beta_lin

In [ ]:
y_lin[:5]

### Basis Functions and Nonlinear Trends


A common trick in finance is to approximate a complicated function with a
linear combination of simpler basis functions: powers, exponentials, or
domain-specific shapes such as splines.


In [ ]:
# Build a richer design matrix with polynomial and sinusoidal basis functions.
X_bf = np.column_stack(
    [
        np.ones_like(x_grid),
        x_grid,
        x_grid**2,
        np.sin(x_grid),
    ]
)
# Solving the least-squares problem returns coefficients for all basis
# functions at
# once, just as in the linear case.
beta_bf, *_ = np.linalg.lstsq(X_bf, y_obs, rcond=None)

In [ ]:
beta_bf

In [ ]:
y_bf = X_bf @ beta_bf

In [ ]:
np.max(np.abs(y_bf - y_clean))

In [ ]:
# Recreate the regression figure from the chapter with the shared notebook
# style.
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(
    x_grid,
    y_clean,
    color="black",
    linewidth=1.5,
    label="True term structure",
)
ax.scatter(
    x_grid,
    y_obs,
    color="tab:gray",
    s=20,
    alpha=0.7,
    label="Noisy observations",
)
ax.plot(
    x_grid,
    y_lin,
    color="tab:orange",
    linestyle="--",
    linewidth=1.25,
    label="Linear regression",
)
ax.plot(
    x_grid,
    y_bf,
    color="tab:blue",
    linestyle="-.",
    linewidth=1.25,
    label="Basis-function regression",
)
ax.set_xlabel("Maturity")
ax.set_ylabel("Value")
ax.set_title("Term-Structure Approximation by Regression")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

### Handling Noisy and Unsorted Data


Least-squares regression does not require equally spaced or sorted input data.


In [ ]:
# The permutation index shuffles positions of the maturities without
# changing their
# values.
perm = rng.permutation(x_grid.shape[0])
x_shuffled = x_grid[perm]
y_shuffled = y_obs[perm]
# Building the design matrix from the shuffled inputs leaves the regression
# problem
# mathematically unchanged.
X_shuffled = np.column_stack(
    [
        np.ones_like(x_shuffled),
        x_shuffled,
        x_shuffled**2,
        np.sin(x_shuffled),
    ]
)
beta_shuffled, *_ = np.linalg.lstsq(X_shuffled, y_shuffled, rcond=None)
# Regression only cares about the rows of the design matrix and target
# vector, not
# about their order.
np.allclose(beta_shuffled, beta_bf)

In [ ]:
# Compute the fitted values on the shuffled grid and sort them only for
# plotting.
y_fit_shuffled = X_shuffled @ beta_shuffled
sort_idx = np.argsort(x_shuffled)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=True)
ax = axes[0]
ax.plot(x_grid, y_clean, color="black", linewidth=1.5, label="True function")
ax.scatter(x_grid, y_obs, color="tab:gray", s=20, alpha=0.7, label="Noisy data")
ax.plot(x_grid, y_bf, color="tab:blue", linewidth=1.25, label="Regression fit")
ax.set_title("Noisy Data (Sorted)")
ax.set_xlabel("x")
ax.set_ylabel("f(x)")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
ax = axes[1]
ax.plot(x_grid, y_clean, color="black", linewidth=1.5, label="True function")
ax.scatter(
    x_shuffled,
    y_shuffled,
    color="tab:gray",
    s=20,
    alpha=0.7,
    label="Noisy, unsorted data",
)
ax.plot(
    x_shuffled[sort_idx],
    y_fit_shuffled[sort_idx],
    color="tab:orange",
    linewidth=1.25,
    label="Regression fit",
)
ax.set_title("Noisy Data (Unsorted)")
ax.set_xlabel("x")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

### Interpolation and Splines


Regression approximates an entire relationship in one step.


In [ ]:
from scipy.interpolate import CubicSpline
# A cubic spline is piecewise polynomial with continuous first and second
# derivatives,
# often used for discount and volatility curves.
spline = CubicSpline(x_grid, y_clean)
# Evaluating the spline on a finer grid yields a smooth curve that visually
# hides the
# underlying knot structure.
x_fine = np.linspace(0.0, 10.0, 201)
# Calling the spline like a function returns interpolated values at arbitrary
# maturities.
y_spline = spline(x_fine)
y_spline[:5]

In [ ]:
from scipy.interpolate import interp1d
# Use a coarse knot set to compare linear and cubic interpolation side by side.
x_knots = np.linspace(0.0, 10.0, 12)
y_knots = term_structure(x_knots)
x_fine_plot = np.linspace(0.0, 10.0, 400)
y_true_plot = term_structure(x_fine_plot)
linear_interp = interp1d(x_knots, y_knots, kind="linear")
y_lin_interp = linear_interp(x_fine_plot)
cubic_interp = CubicSpline(x_knots, y_knots)
y_cubic_interp = cubic_interp(x_fine_plot)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.0), sharey=True)
ax = axes[0]
ax.plot(
    x_fine_plot,
    y_true_plot,
    color="black",
    linewidth=1.5,
    label="True function",
)
ax.plot(
    x_fine_plot,
    y_lin_interp,
    color="tab:orange",
    linewidth=1.25,
    label="Linear spline",
)
ax.scatter(x_knots, y_knots, color="tab:blue", s=25, zorder=3, label="Knots")
ax.set_title("Linear Spline Interpolation")
ax.set_xlabel("Maturity")
ax.set_ylabel("Value")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
ax = axes[1]
ax.plot(
    x_fine_plot,
    y_true_plot,
    color="black",
    linewidth=1.5,
    label="True function",
)
ax.plot(
    x_fine_plot,
    y_cubic_interp,
    color="tab:green",
    linewidth=1.25,
    label="Cubic spline",
)
ax.scatter(x_knots, y_knots, color="tab:blue", s=25, zorder=3, label="Knots")
ax.set_title("Cubic Spline Interpolation")
ax.set_xlabel("Maturity")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

## Root Finding and Convex Optimization


Another family of core tasks is solving equations and optimization problems.


### Solving for Yields via Root Finding


Consider a simple zero-coupon bond with face value 100 and maturity in $T$
years.


In [ ]:
from scipy import optimize
face = 100.0
T = 5.0
price_0 = 88.0
def bond_pricing_error(yield_cc: float) -> float:
    """Pricing error for a continuously compounded yield."""
    model_price = face * np.exp(-yield_cc * T)
    return model_price - price_0
# Use a bracketed scalar root finder to recover the yield from the observed
# price.
result_y = optimize.root_scalar(
    bond_pricing_error,
    bracket=(0.0, 0.2),
    method="brentq",
)

### A Simple Mean–Variance Portfolio Problem


Convex optimization problems appear throughout portfolio construction.


In [ ]:
mu = np.array([0.06, 0.08])
sigma = np.array([0.15, 0.25])
rho = 0.4
cov = np.array(
    [
        [sigma[0]**2, rho * sigma[0] * sigma[1]],
        [rho * sigma[0] * sigma[1], sigma[1]**2],
    ]
)
# Target a portfolio return of 7% per year under long-only constraints.
target_return = 0.07
def portfolio_variance(weights: np.ndarray) -> float:
    # `weights @ cov @ weights` computes the quadratic form $w^\top \Sigma w$.
    return float(weights @ cov @ weights)
def return_constraint(weights: np.ndarray) -> float:
    return float(weights @ mu - target_return)
cons = (
    {"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
    {"type": "eq", "fun": return_constraint},
)
bounds = ((0.0, 1.0), (0.0, 1.0))
w0 = np.array([0.5, 0.5])
opt_res = optimize.minimize(
    portfolio_variance,
    w0,
    method="SLSQP",
    bounds=bounds,
    constraints=cons,
)

In [ ]:
opt_res.x, opt_res.success

In [ ]:
portfolio_variance(opt_res.x)

### Global and Local Optimization in Two Dimensions


Optimization problems in more than two dimensions are hard to visualize, but
it is still useful to develop intuition from low-dimensional examples.


In [ ]:
def objective(p: np.ndarray) -> np.ndarray:
    x, y = p
    return np.sin(x) + 0.05 * x**2 + np.sin(y) + 0.05 * y**2
# Start with a coarse global grid search before refining the best candidate
# locally.
grid_ranges = ((-10.0, 10.0, 0.5), (-10.0, 10.0, 0.5))
brute_min, f_min, _, _ = optimize.brute(
    objective,
    ranges=grid_ranges,
    full_output=True,
    finish=None,
)

In [ ]:
brute_min, f_min

In [ ]:
local_res = optimize.minimize(
    objective,
    brute_min,
    method="Nelder-Mead",
)

In [ ]:
local_res.x, local_res.fun

In [ ]:
# Evaluate the two-parameter objective on a grid and render the surface used
# in the chapter.
x_surface = np.linspace(-10.0, 10.0, 80)
y_surface = np.linspace(-10.0, 10.0, 80)
X_surface, Y_surface = np.meshgrid(x_surface, y_surface)
Z_surface = objective((X_surface, Y_surface))
fig = plt.figure(figsize=(7.5, 4.5))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(
    X_surface,
    Y_surface,
    Z_surface,
    rstride=2,
    cstride=2,
    cmap="coolwarm",
    linewidth=0.4,
    antialiased=True,
)
ax.set_title("Two-Parameter Objective Function")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("f(x, y)")
fig.colorbar(surf, shrink=0.6, aspect=16)
fig.tight_layout()
plt.show()

## Numerical Integration


Integration underpins many valuation formulas.


### Integrating the Standard Normal Density


As a warm-up, integrate the standard normal probability density function (PDF)
over the real line.


In [ ]:
from math import exp, sqrt, pi
from scipy import integrate
def normal_pdf(x: float) -> float:
    # The standard normal PDF is implemented directly from the
    # analytical formula
    # without relying on external libraries.
    return (1.0 / sqrt(2.0 * pi)) * exp(-0.5 * x * x)
# The tiny error estimate confirms that the numerical integral matches the
# analytical
# value.
value, error = integrate.quad(normal_pdf, -np.inf, np.inf)
value, error

In [ ]:
# Visualize the integral as the shaded area under the standard normal density.
x_plot = np.linspace(-4.0, 4.0, 500)
y_plot = np.array([normal_pdf(xi) for xi in x_plot])
a, b = -1.0, 1.0
mask = (x_plot >= a) & (x_plot <= b)
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.plot(
    x_plot,
    y_plot,
    color="tab:blue",
    linewidth=1.5,
    label="Standard normal PDF",
)
ax.fill_between(
    x_plot[mask],
    0.0,
    y_plot[mask],
    color="tab:blue",
    alpha=0.25,
    label="Area between a and b",
)
ax.axvline(a, color="tab:gray", linestyle="--", linewidth=1.0)
ax.axvline(b, color="tab:gray", linestyle="--", linewidth=1.0)
ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title("Integral as Shaded Area Under the Normal Density")
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(loc="best")
fig.tight_layout()
plt.show()

### Option Pricing as an Integral


For a Black–Scholes European call option with strike $K$, maturity $T$,
risk-free rate $r$, and lognormally distributed terminal price $S_T$, the
undiscounted payoff is $\max(S_T - K, 0)$.


In [ ]:
s0 = 100.0
K = 100.0
r = 0.02
sigma = 0.2
T = 1.0
# Parameterise the lognormal terminal-price distribution implied by GBM.
m = np.log(s0) + (r - 0.5 * sigma**2) * T
v = sigma * np.sqrt(T)
def lognormal_pdf(s: float) -> float:
    if s <= 0.0:
        return 0.0
    z = (np.log(s) - m) / v
    return (1.0 / (s * v * np.sqrt(2.0 * np.pi))) * np.exp(-0.5 * z * z)
def call_payoff(s: float) -> float:
    return max(s - K, 0.0)
def integrand(s: float) -> float:
    return np.exp(-r * T) * call_payoff(s) * lognormal_pdf(s)
# Integrate over a wide but finite support to approximate the call price.
price_int, err_int = integrate.quad(integrand, 0.0, 5.0 * s0)

## Symbolic Mathematics with SymPy


Symbolic computation is useful when you need exact derivatives, closed-form
solutions, or simplified expressions.


### Solving Nonlinear Equations Symbolically


Suppose you want to solve for the continuously compounded yield $y$ of a
zero-coupon bond analytically, as in the earlier root-finding example.


In [ ]:
# Importing `sympy` as `sp` follows the conventional shorthand used in most
# documentation and examples.
import sympy as sp
# The symbols represent the unknown yield `y` and the positive parameters
# maturity
# `T`, price `P`, and face value `F`.
y, T_sym, P_sym, F_sym = sp.symbols("y T P F", positive=True)
# `bond_eq` encodes the pricing relation $F \mathrm{e}^{-y T} = P$ in
# symbolic form.
bond_eq = sp.Eq(F_sym * sp.exp(-y * T_sym), P_sym)
# `sp.solve(..., y)[0]` isolates `y` and yields the analytical solution $y =
# -\log(P/F) / T$.
y_solution = sp.solve(bond_eq, y)[0]
y_solution

### Differentiation and Greeks


For derivatives pricing you often need sensitivities (“Greeks”) with respect
to model parameters.


In [ ]:
# Symbolically differentiate a Black-Scholes-style call-price expression.
S, K_sym, r_sym, sigma_sym, T_sym = sp.symbols(
    "S K r sigma T",
    positive=True,
)
from sympy import exp, log, sqrt
d1 = (log(S / K_sym) + (r_sym + 0.5 * sigma_sym**2) * T_sym) / (
    sigma_sym * sqrt(T_sym)
)
N = sp.Function("N")
call_sym = S * N(d1) - K_sym * exp(-r_sym * T_sym) * N(
    d1 - sigma_sym * sqrt(T_sym)
)
delta_sym = sp.diff(call_sym, S)

In [ ]:
delta_sym

### From Symbolic to Numerical Functions


`SymPy` expressions can be turned into fast numerical functions with
`sp.lambdify()`.


In [ ]:
x = sp.symbols("x")
f_sym = sp.exp(-x) * sp.sin(x)
f_num = sp.lambdify(x, f_sym, modules=["numpy"])
x_vals = np.linspace(0.0, 5.0, 6)

In [ ]:
f_num(x_vals)

## Figure Generation (Optional)
Run the chapter's figure scripts under `code/figures/` to regenerate the PNG
files under `assets/figures/`.


In [ ]:
import runpy

scripts = [
    "../code/figures/ch12_function_surface.py",
    "../code/figures/ch12_normal_integral_area.py",
    "../code/figures/ch12_regression_noisy_unsorted.py",
    "../code/figures/ch12_spline_interpolation.py",
    "../code/figures/ch12_term_structure_regression.py",
]

for script in scripts:
    try:
        runpy.run_path(script, run_name="__main__")
        print(f"OK: {script}")
    except ModuleNotFoundError as e:
        print(f"Skipping {script}: missing dependency ({e.name}).")
    except Exception as e:
        print(f"Failed {script}: {type(e).__name__}: {e}")


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
